In [49]:
import os
import json
import time
from typing import List, Dict, Any
from docx import Document
from requests.exceptions import RequestException
from langchain_gigachat.chat_models import GigaChat
from langchain_core.messages import HumanMessage


SYSTEM_PROMPT = """
[СИСТЕМНЫЙ ПРОМПТ]
Твоя роль – обработчик новостей.
Твоя задача – из предоставленного списка новостей, которые относятся к темам "Макроэкономика России", "Международные экономические новости", "Валюты",  "Инвестиции". Также они могут выступать как рекомендации:


[КРИТЕРИИ ВЫВОДА]
1. Выводи только новости без лишних слов и символов.
2. Формат ответа – JSON-массив с полями: title
3. Отвечай только валидным JSON без дополнительного текста.

[ОБРАЗЕЦ ВЫВОДА]

[
    {"title": "Новость 1"},
]
"""


In [50]:
model = GigaChat(
    credentials="GIGACHAT_CREDENTIALS_REMOVED",
    scope="GIGACHAT_API_CORP",
    model="GigaChat-2-MAX",
    verify_ssl_certs=False
)

In [51]:
file_path = "quash.docx"
def read_docx_paragraphs(file_path: str) -> List[str]:
    """Читает все непустые параграфы из DOCX-файла."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Файл {file_path} не найден")
    
    doc = Document(file_path)
    paragraphs = []
    for para in doc.paragraphs:
        text = (para.text or "").strip()
        if text:
            paragraphs.append(text)
    return paragraphs


In [52]:
def extract_impactful_news(paragraphs: List[str], output_file: str = 'impactful_news.json'):
    # Собираем все новости в единый текст
    combined_news = "\n".join(f"- {p}" for p in paragraphs)
    prompt = SYSTEM_PROMPT + f"\n\n[ДАННЫЕ]\n{combined_news}\n\nОТВЕТ:"

    messages = [HumanMessage(content=prompt)]
    news_list: List[Dict[str, Any]] = []

    for attempt in range(3):
        try:
            response = model.invoke(messages)
            raw = response.content or ""
            content = raw.strip()

            # Удаляем BOM
            if content.startswith('\ufeff'):
                content = content.lstrip('\ufeff')

            # Вырезаем JSON между первой '[' и последней ']'
            start = content.find('[')
            end = content.rfind(']') + 1
            if start != -1 and end != -1:
                content = content[start:end]

            # Парсим JSON
            news_list = json.loads(content)
            print(f"JSON успешно разобран: {len(news_list)} элементов")
            break

        except json.JSONDecodeError as e:
            print(f"Ошибка парсинга JSON (попытка {attempt+1}): {e}")
            print(f"Сырой ответ после очистки: {content!r}")
            time.sleep(2 ** attempt)

        except Exception as e:
            print(f"Непредвиденная ошибка: {e}")
            break

    if not news_list:
        news_list = [{"error": "Не удалось извлечь новости"}]

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(news_list, f, ensure_ascii=False, indent=4)
    print(f"Отобранные новости сохранены в «{output_file}».")


In [53]:
if __name__ == "__main__":
    file_path = "quash.docx"  # Путь к вашему файлу
    paragraphs = read_docx_paragraphs(file_path)
    print(f"Найдено абзацев: {len(paragraphs)}")
    extract_impactful_news(paragraphs, output_file='impactful_news.json')


Найдено абзацев: 130
JSON успешно разобран: 32 элементов
Отобранные новости сохранены в «impactful_news.json».
